# OWCS Result Screen Extraction Pipeline

This notebook contains the AI-assisted data extraction and validation pipeline used for the OWCS Korea Stat Lab.

The pipeline converts broadcast result-screen screenshots into structured player-level statistics using the following process:

**Result Screens → AI-Assisted Extraction → Deterministic Validation → Structured Dataset → Player Aggregation → Per-10 Statistics**

AI-generated output is treated as an intermediate extraction result rather than ground truth. Extracted records must pass deterministic validation checks before entering the processed dataset.

### Pipeline Overview

1. Load a canonical roster reference
2. Extract scoreboard data from each result screen
3. Parse the model response into a fixed schema
4. Validate player identities, numeric fields, playtime, and scoreboard consistency
5. Exclude failed extractions
6. Preserve source-file traceability
7. Aggregate validated records by player
8. Calculate accumulated and Per-10-minute statistics

In [1]:
import os
import re
import csv
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import google.generativeai as genai


# ============================================================
# API CONFIGURATION
# ============================================================

# Set GEMINI_API_KEY in your environment before running.
#
# Example:
#   export GEMINI_API_KEY="your_api_key_here"
#
# Never hardcode API keys directly in this notebook.

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. "
        "Set it as an environment variable before running this notebook."
    )

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(MODEL_NAME)


# ============================================================
# INPUT / OUTPUT PATHS
# ============================================================

# Replace these example paths with your local file locations.

# Folder containing result-screen screenshots.
IMAGE_DIR = Path("path/to/result_screens")

# Workbook containing the canonical TEAM INFO sheet.
ROSTER_XLSX = Path("path/to/roster_reference.xlsx")

# Pipeline outputs.
OUTPUT_DIR = Path("output")

PARSED_XLSX = OUTPUT_DIR / "validated_results.xlsx"
CLEAN_CSV = OUTPUT_DIR / "clean_raw_data.csv"
REVIEW_CSV = OUTPUT_DIR / "validation_report.csv"
CHECKPOINT_CSV = OUTPUT_DIR / "checkpoint.csv"
MASTER_XLSX = OUTPUT_DIR / "player_statistics.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# PIPELINE SETTINGS
# ============================================================

MAX_RETRIES = 3
REQUEST_DELAY_SEC = 1

CHUNK_SIZE = 6
CHUNK_COOLDOWN_SEC = 15

MIN_PER10_MINUTES = 30

/opt/anaconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/var/folders/kc/t3sn0bvn6djgck1vl99fwf1c0000gn/T/ipykernel_69002/2663694738.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


RuntimeError: GEMINI_API_KEY is not set. Set it as an environment variable before running this notebook.

## Extraction Schema

Each result screen is expected to contain exactly 10 player records.

The extraction model returns the following fields:

| Field | Description |
|---|---|
| Team | Team abbreviation shown on screen |
| Player | Player ID shown on screen |
| Position | Role shown for the player on that map |
| Map | Map name |
| Map Type | Game mode |
| Elim | Eliminations |
| Death | Deaths |
| Assists | Assists |
| Damage | Damage dealt |
| Heal | Healing |
| Mitigated | Damage mitigated |
| Playtime | Map playtime |

Player identity and team membership are validated against a canonical roster after extraction.

In [1]:
COLUMNS = [
    "Team",
    "Player",
    "Position",
    "Map",
    "Map Type",
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
    "Playtime",
]

NUMERIC_COLS = [
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
]

VALID_POSITIONS = {
    "TANK",
    "DPS",
    "SPT",
}

VALID_MAP_TYPES = {
    "Control",
    "Hybrid",
    "Escort",
    "Flashpoint",
    "Push",
    "Clash",
}


PROMPT = """
This image is an Overwatch esports result/scoreboard screen.

Extract exactly the 10 player rows and return CSV only.
Do not include a header, markdown, or explanation.

Columns, in this exact order:
Team,Player,Position,Map,Map Type,Elim,Death,Assists,Damage,Heal,Mitigated,Playtime

Rules:
- Return exactly 10 rows, one per player.
- Team: use the team abbreviation shown on screen.
- Player: copy the player ID exactly as displayed. Do not guess or autocorrect.
- Position: TANK, DPS, or SPT only.
- Map: use the map name shown on screen.
- Map Type: Control, Hybrid, Escort, Flashpoint, Push, or Clash.
- Carefully map scoreboard E/A/D values to Elim/Assists/Death.
- Damage, Heal, and Mitigated must contain digits only.
- Do not include thousands separators.
- Playtime must use HH:MM:SS format.
- All 10 players should have the same map and playtime.
- If a value cannot be read confidently, leave that field empty.
- Never invent a value.
""".strip()

FileNotFoundError: [Errno 2] No such file or directory: 'OWCS_PARSED_RESULT_V5.xlsx'

## Canonical Roster Reference

The `TEAM INFO` sheet is used as the canonical reference for player identity and team membership.

The roster reference is also used to store each player's primary roster role and detailed analytical role.

Map-level `Position` is preserved separately because a player may compete in a different role during an individual map.

In [ ]:
def load_roster(path):
    """
    Load canonical player information from the TEAM INFO sheet.

    Expected columns:
    Team Name | Shortened | Player Name | Player Position | Detailed
    """

    if not path.exists():
        print(
            f"⚠️ Roster file not found: {path}. "
            "Player/team validation will be limited."
        )
        return {}

    team_info = pd.read_excel(
        path,
        sheet_name="TEAM INFO"
    )

    roster = {}

    current_team = None
    current_short = None

    for _, row in team_info.iterrows():

        values = row.tolist()

        team = values[0] if len(values) > 0 else None
        short = values[1] if len(values) > 1 else None
        player = values[2] if len(values) > 2 else None
        position = values[3] if len(values) > 3 else None
        detailed = values[4] if len(values) > 4 else None

        if pd.notna(team) and pd.notna(short):
            current_team = str(team).strip()
            current_short = str(short).strip()

        if pd.notna(player):

            player_id = str(player).strip()

            roster[player_id] = {
                "Team": current_short,
                "Roster Position": (
                    str(position).strip()
                    if pd.notna(position)
                    else None
                ),
                "Detailed Position": (
                    str(detailed).strip()
                    if pd.notna(detailed)
                    else None
                ),
            }

    return roster


ROSTER = load_roster(ROSTER_XLSX)

print(
    f"Loaded {len(ROSTER)} players "
    "from TEAM INFO."
)

In [ ]:
def clean_model_text(text):
    """Remove optional Markdown code fences."""

    text = (text or "").strip()

    text = re.sub(
        r"^```(?:csv)?\s*",
        "",
        text,
        flags=re.I,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    return text.strip()


def parse_csv_response(text):
    """
    Parse the model response into the expected schema.

    Lines with an unexpected number of fields are retained
    separately as malformed lines for validation.
    """

    rows = []
    bad_lines = []

    reader = csv.reader(
        clean_model_text(text).splitlines()
    )

    for line_no, items in enumerate(
        reader,
        start=1,
    ):

        items = [
            item.strip()
            for item in items
        ]

        if not any(items):
            continue

        if len(items) != len(COLUMNS):

            bad_lines.append(
                (line_no, items)
            )

            continue

        rows.append(
            dict(zip(COLUMNS, items))
        )

    return rows, bad_lines


def parse_number(value):
    """
    Convert an extracted numeric field to int or float.

    Invalid values return None instead of being silently
    converted to zero.
    """

    s = str(value).strip().replace(",", "")

    if not s:
        return None

    if not re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        s,
    ):
        return None

    return (
        float(s)
        if "." in s
        else int(s)
    )


def normalize_time(value):
    """Normalize playtime to HH:MM:SS."""

    s = str(value).strip()

    if not s:
        return None

    parts = s.split(":")

    try:

        if len(parts) == 2:
            h = 0
            m, sec = map(int, parts)

        elif len(parts) == 3:
            h, m, sec = map(int, parts)

        else:
            return None

        if (
            min(h, m, sec) < 0
            or m >= 60
            or sec >= 60
        ):
            return None

        return f"{h:02d}:{m:02d}:{sec:02d}"

    except ValueError:
        return None

## Deterministic Validation

AI output is not treated as ground truth.

Each extracted result screen is checked for:

- malformed CSV rows
- exactly 10 player records
- duplicate player IDs
- unknown player IDs
- invalid numeric fields
- invalid playtime
- invalid or unusual categorical values
- inconsistent map information
- inconsistent playtime across the result screen
- team membership against the canonical roster

Validation produces one of three statuses:

- **PASS** — no detected issues
- **WARNING** — usable record with a detectable non-fatal inconsistency
- **FAIL** — record excluded from the clean dataset

Both PASS and WARNING records remain traceable to their original source screenshot.

In [ ]:
def validate_image_rows(
    rows,
    bad_lines,
    filename,
):
    """
    Validate and normalize one extracted scoreboard.

    FAIL records are excluded from the clean dataset.
    WARNING records remain available for review.
    """

    issues = []
    fatal = False


    # --------------------------------------------------------
    # Structural validation
    # --------------------------------------------------------

    if bad_lines:

        issues.append(
            f"{len(bad_lines)} malformed CSV line(s)"
        )

        fatal = True


    if len(rows) != 10:

        issues.append(
            f"expected 10 rows, got {len(rows)}"
        )

        fatal = True


    # --------------------------------------------------------
    # Player validation
    # --------------------------------------------------------

    players = [
        row["Player"].strip()
        for row in rows
    ]

    dupes = sorted({
        player
        for player in players
        if player
        and players.count(player) > 1
    })

    if dupes:

        issues.append(
            "duplicate player(s): "
            + ", ".join(dupes)
        )

        fatal = True


    unknown = sorted({
        player
        for player in players
        if player
        and ROSTER
        and player not in ROSTER
    })

    if unknown:

        issues.append(
            "unknown player(s): "
            + ", ".join(unknown)
        )

        fatal = True


    # --------------------------------------------------------
    # Row-level validation
    # --------------------------------------------------------

    normalized = []

    for idx, row in enumerate(
        rows,
        start=1,
    ):

        record = dict(row)

        player = record["Player"].strip()

        row_issues = []


        # Numeric fields
        for col in NUMERIC_COLS:

            value = parse_number(
                record[col]
            )

            if value is None:
                row_issues.append(
                    f"{col} invalid"
                )

            record[col] = value


        # Playtime
        playtime = normalize_time(
            record["Playtime"]
        )

        if playtime is None:
            row_issues.append(
                "Playtime invalid"
            )

        record["Playtime"] = playtime


        # Position
        if record["Position"] not in VALID_POSITIONS:

            row_issues.append(
                f"Position invalid: "
                f"{record['Position']}"
            )


        # Map type
        if record["Map Type"] not in VALID_MAP_TYPES:

            row_issues.append(
                f"Map Type unusual: "
                f"{record['Map Type']}"
            )


        # ----------------------------------------------------
        # Canonical roster cross-reference
        # ----------------------------------------------------

        if player in ROSTER:

            canonical = ROSTER[player]

            canonical_team = canonical.get(
                "Team"
            )

            if (
                canonical_team
                and record["Team"]
                != canonical_team
            ):

                row_issues.append(
                    "Team corrected "
                    f"{record['Team']}→"
                    f"{canonical_team}"
                )

            # Canonical team membership wins.
            record["Team"] = (
                canonical_team
                or record["Team"]
            )

            # Preserve map-level Position.
            #
            # A player may compete in a role different from
            # their primary roster classification.

            roster_position = canonical.get(
                "Roster Position"
            )

            if (
                roster_position
                and record["Position"]
                != roster_position
            ):

                row_issues.append(
                    "Map role differs from roster role "
                    f"({record['Position']} vs "
                    f"{roster_position})"
                )

            record["Roster Position"] = (
                roster_position
            )

            record["Detailed Position"] = (
                canonical.get(
                    "Detailed Position"
                )
            )

        else:

            record["Roster Position"] = None
            record["Detailed Position"] = None


        # Required values
        if (
            any(
                record[col] is None
                for col in NUMERIC_COLS
            )
            or record["Playtime"] is None
        ):
            fatal = True


        if row_issues:

            issues.append(
                f"row {idx} "
                f"({player or 'UNKNOWN'}): "
                + "; ".join(row_issues)
            )


        normalized.append(record)


    # --------------------------------------------------------
    # Scoreboard-level consistency
    # --------------------------------------------------------

    for col in [
        "Map",
        "Map Type",
        "Playtime",
    ]:

        values = {
            str(record.get(col))
            for record in normalized
            if record.get(col)
            not in (None, "")
        }

        if len(values) > 1:

            issues.append(
                f"inconsistent {col}: "
                f"{sorted(values)}"
            )

            fatal = True


    status = (
        "FAIL"
        if fatal
        else (
            "WARNING"
            if issues
            else "PASS"
        )
    )

    return normalized, status, issues

In [ ]:
def extract_one_image(path):
    """
    Extract and validate one result-screen screenshot.
    """

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1,
    ):

        try:

            uploaded = genai.upload_file(
                path=str(path)
            )

            response = model.generate_content(
                [uploaded, PROMPT]
            )

            rows, bad_lines = (
                parse_csv_response(
                    response.text
                )
            )

            normalized, status, issues = (
                validate_image_rows(
                    rows,
                    bad_lines,
                    path.name,
                )
            )

            return (
                normalized,
                status,
                issues,
                response.text,
            )


        except Exception as error:

            last_error = error

            if attempt < MAX_RETRIES:

                wait = 2 ** (
                    attempt - 1
                )

                print(
                    f"    retry "
                    f"{attempt}/{MAX_RETRIES} "
                    f"after error: "
                    f"{error} ({wait}s)"
                )

                time.sleep(wait)


    return (
        [],
        "FAIL",
        [
            "API/parsing error after "
            f"{MAX_RETRIES} attempts: "
            f"{last_error}"
        ],
        "",
    )

In [ ]:
clean_rows = []
report = []
processed_files = set()


# ============================================================
# RESUME FROM CHECKPOINT
# ============================================================

if CHECKPOINT_CSV.exists():

    checkpoint = pd.read_csv(
        CHECKPOINT_CSV
    )

    if "Source File" in checkpoint.columns:

        processed_files = set(
            checkpoint["Source File"]
            .dropna()
            .astype(str)
        )

        clean_rows = (
            checkpoint
            .to_dict("records")
        )

        print(
            "Resuming from checkpoint: "
            f"{len(processed_files)} "
            "image(s) already accepted."
        )


# ============================================================
# DISCOVER SCREENSHOTS
# ============================================================

if not IMAGE_DIR.exists():

    raise FileNotFoundError(
        f"Image folder not found: "
        f"{IMAGE_DIR}"
    )


image_files = sorted([
    path
    for path in IMAGE_DIR.iterdir()
    if path.suffix.lower()
    in {".png", ".jpg", ".jpeg"}
])


print(
    f"📸 {len(image_files)} "
    "screenshots found."
)


pending = [
    path
    for path in image_files
    if path.name not in processed_files
]


# ============================================================
# RUN EXTRACTION
# ============================================================

for idx, path in enumerate(
    pending,
    start=1,
):

    print(
        f"[{idx}/{len(pending)}] "
        f"{path.name}"
    )


    rows, status, issues, raw_response = (
        extract_one_image(path)
    )


    report.append({
        "Source File": path.name,
        "Status": status,
        "Rows Parsed": len(rows),
        "Issues": " | ".join(issues),
        "Checked At": (
            datetime.now()
            .isoformat(timespec="seconds")
        ),
    })


    # PASS and WARNING records enter the clean dataset.
    # FAIL records are excluded.

    if status != "FAIL":

        for row in rows:

            row["Source File"] = path.name
            row["Validation Status"] = status

            clean_rows.append(row)


        pd.DataFrame(
            clean_rows
        ).to_csv(
            CHECKPOINT_CSV,
            index=False,
            encoding="utf-8-sig",
        )


        print(
            f"    {status}: "
            f"accepted {len(rows)} rows"
        )


    else:

        print(
            "    FAIL: excluded "
            "from clean dataset"
        )

        for issue in issues:
            print(
                "      -",
                issue,
            )


    time.sleep(
        REQUEST_DELAY_SEC
    )


    if (
        idx % CHUNK_SIZE == 0
        and idx < len(pending)
    ):

        print(
            "⏳ chunk cooldown: "
            f"{CHUNK_COOLDOWN_SEC}s"
        )

        time.sleep(
            CHUNK_COOLDOWN_SEC
        )

In [ ]:
clean_df = pd.DataFrame(
    clean_rows
)

report_df = pd.DataFrame(
    report
)


if not clean_df.empty:

    clean_df.to_csv(
        CLEAN_CSV,
        index=False,
        encoding="utf-8-sig",
    )


if not report_df.empty:

    report_df.to_csv(
        REVIEW_CSV,
        index=False,
        encoding="utf-8-sig",
    )


with pd.ExcelWriter(
    PARSED_XLSX
) as writer:

    clean_df.to_excel(
        writer,
        sheet_name="Clean Raw Data",
        index=False,
    )

    report_df.to_excel(
        writer,
        sheet_name="Validation Report",
        index=False,
    )


print(
    f"✅ Clean rows: "
    f"{len(clean_df)}"
)

print(
    f"📋 Validation report: "
    f"{REVIEW_CSV}"
)

print(
    f"📗 Validated workbook: "
    f"{PARSED_XLSX}"
)

# Statistical Processing

The following step generates accumulated and Per-10-minute player statistics from the validated map-level dataset.

Both accumulated and Per-10 statistics are derived from the same clean source to maintain consistency.

Player ID is used as the primary aggregation key.

Map-level `Position` is intentionally not treated as part of player identity because some players may compete in multiple roles during the competition.

In [ ]:
# ============================================================
# LOAD VALIDATED DATA
# ============================================================

raw = pd.read_excel(
    PARSED_XLSX,
    sheet_name="Clean Raw Data",
)


numeric_cols = [
    "Elim",
    "Death",
    "Assists",
    "Damage",
    "Heal",
    "Mitigated",
]


for col in numeric_cols:

    raw[col] = pd.to_numeric(
        raw[col],
        errors="raise",
    )


# ============================================================
# PLAYTIME
# ============================================================

def time_to_seconds(value):
    """Convert HH:MM:SS or MM:SS to seconds."""

    if pd.isna(value):

        raise ValueError(
            "Missing Playtime in "
            "accepted clean data."
        )


    parts = str(value).strip().split(":")


    if len(parts) == 3:

        h, m, sec = map(
            int,
            parts,
        )


    elif len(parts) == 2:

        h = 0

        m, sec = map(
            int,
            parts,
        )


    else:

        raise ValueError(
            f"Invalid Playtime: {value}"
        )


    if (
        min(h, m, sec) < 0
        or m >= 60
        or sec >= 60
    ):

        raise ValueError(
            f"Invalid Playtime: {value}"
        )


    return (
        h * 3600
        + m * 60
        + sec
    )


raw["Playtime_Seconds"] = (
    raw["Playtime"]
    .apply(time_to_seconds)
)


# ============================================================
# PLAYER IDENTITY
# ============================================================

# Player ID is the aggregation key.
#
# Map-level Position is intentionally excluded because
# players may compete in multiple roles.

identity_cols = ["Team"]


if "Roster Position" in raw.columns:
    identity_cols.append(
        "Roster Position"
    )


if "Detailed Position" in raw.columns:
    identity_cols.append(
        "Detailed Position"
    )


identity_check = (
    raw.groupby("Player")[
        identity_cols
    ]
    .nunique(dropna=False)
)


problem_players = (
    identity_check[
        (identity_check > 1)
        .any(axis=1)
    ]
)


if not problem_players.empty:

    raise ValueError(
        "Canonical identity conflict "
        "for player(s): "
        + ", ".join(
            problem_players
            .index
            .astype(str)
        )
    )


identity = (
    raw.groupby(
        "Player",
        as_index=False,
    )[identity_cols]
    .first()
)


# ============================================================
# ACCUMULATED STATISTICS
# ============================================================

stats = (
    raw.groupby(
        "Player",
        as_index=False,
    )[
        numeric_cols
        + ["Playtime_Seconds"]
    ]
    .sum()
)


total = identity.merge(
    stats,
    on="Player",
    how="inner",
)


total["Playtime_Min"] = (
    total["Playtime_Seconds"]
    / 60
)


front = [
    "Team",
    "Player",
]


for col in [
    "Roster Position",
    "Detailed Position",
]:

    if col in total.columns:
        front.append(col)


total = total[
    front
    + numeric_cols
    + [
        "Playtime_Seconds",
        "Playtime_Min",
    ]
]


# ============================================================
# PER-10-MINUTE STATISTICS
# ============================================================

per10 = total[
    front
    + [
        "Playtime_Seconds",
        "Playtime_Min",
    ]
].copy()


for col in numeric_cols:

    per10[f"{col} / 10"] = (
        total[col]
        / total["Playtime_Seconds"]
        * 600
    ).round(2)


# Keep all calculated values in the processed dataset.
#
# The eligibility flag is used by rankings and
# visualizations to exclude small samples.

per10["Eligible_30min"] = (
    per10["Playtime_Min"]
    >= MIN_PER10_MINUTES
)


# ============================================================
# EXPORT
# ============================================================

with pd.ExcelWriter(
    MASTER_XLSX
) as writer:

    total.to_excel(
        writer,
        sheet_name="TOTAL STATISTICS",
        index=False,
    )

    per10.to_excel(
        writer,
        sheet_name="Per10 STATISTICS",
        index=False,
    )

    raw.to_excel(
        writer,
        sheet_name="Clean Raw Data",
        index=False,
    )


print(
    f"✅ {len(total)} players aggregated "
    "from the same validated raw dataset."
)

print(
    "✅ Per10 eligibility threshold: "
    f"{MIN_PER10_MINUTES} minutes."
)

print(
    f"📗 Master workbook: "
    f"{MASTER_XLSX}"
)